In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "../Data"   # change to "data" if needed
MODEL_DIR = BASE_DIR / "app" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(BASE_DIR)
print(DATA_DIR.exists())
print(MODEL_DIR)

/Users/m.mughees/Desktop/Pi-515-2026/src
True
/Users/m.mughees/Desktop/Pi-515-2026/src/app/models


In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (2099831, 26)
Test shape: (10000, 26)


In [5]:
base_features = [
    "num__latitude",
    "num__longitude",
    "num__total_precipitation",
    "num__runoff",
    "num__total_evaporation",
    "num__potential_evaporation",
    "num__2m_dewpoint_temperature",
    "num__2m_temperature",
    "num__year",
    "num__month",
    "num__day",
    "cat__DistrictNa_Central",
    "cat__DistrictNa_East Central",
    "cat__DistrictNa_North Central",
    "cat__DistrictNa_Northeast",
    "cat__DistrictNa_Northwest",
    "cat__DistrictNa_South Central",
    "cat__DistrictNa_Southeast",
    "cat__DistrictNa_Southwest",
    "cat__DistrictNa_West Central",
]

soil_temp_target = "num__soil_temperature_level_1"
soil_moisture_target = "Soil_Moisture"

In [6]:
soil_temp_model = joblib.load(MODEL_DIR / "soil_temperature_model.joblib")
print("Loaded soil temperature model.")

Loaded soil temperature model.


In [7]:
X_train_temp = train_df[base_features]
X_test_temp = test_df[base_features]

train_pred_temp = soil_temp_model.predict(X_train_temp)
test_pred_temp = soil_temp_model.predict(X_test_temp)

print(train_pred_temp[:5])
print(test_pred_temp[:5])

[-1.37327494  0.97259278  0.3751716  -0.61975059  0.96776712]
[ 0.54670404  0.32466328 -1.10865339 -1.38588994 -1.64209133]


In [8]:
train_stage2 = train_df.copy()
test_stage2 = test_df.copy()

train_stage2["predicted_soil_temp"] = train_pred_temp
test_stage2["predicted_soil_temp"] = test_pred_temp

moisture_features = base_features + ["predicted_soil_temp"]

X_train = train_stage2[moisture_features]
y_train = train_stage2[soil_moisture_target]

X_test = test_stage2[moisture_features]
y_test = test_stage2[soil_moisture_target]

print(X_train.shape, X_test.shape)

(2099831, 21) (10000, 21)


In [9]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(name, y_true, y_pred):
    metrics = {
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }
    print(f"\n{name}")
    print("-" * 30)
    for k, v in metrics.items():
        print(f"{k.upper()}: {v:.4f}")
    return metrics

def add_irrigation_category(values: pd.Series) -> pd.Series:
    low_thresh = values.quantile(0.33)
    high_thresh = values.quantile(0.66)

    def label(v):
        if v <= low_thresh:
            return "HIGH"
        elif v <= high_thresh:
            return "MEDIUM"
        return "LOW"

    return values.apply(label)

In [10]:
soil_moisture_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=14,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

soil_moisture_model.fit(X_train, y_train)

RandomForestRegressor(max_depth=14, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=300, n_jobs=-1, random_state=42)

In [11]:
train_pred = soil_moisture_model.predict(X_train)
test_pred = soil_moisture_model.predict(X_test)

train_metrics = evaluate("Soil Moisture Train", y_train, train_pred)
test_metrics = evaluate("Soil Moisture Test", y_test, test_pred)


Soil Moisture Train
------------------------------
RMSE: 0.0495
MAE: 0.0362
R2: 0.6658

Soil Moisture Test
------------------------------
RMSE: 0.0489
MAE: 0.0360
R2: 0.6648


In [12]:
results_df = pd.DataFrame({
    "actual_soil_moisture": y_test.values,
    "predicted_soil_moisture": test_pred
})

results_df.head(10)

,actual_soil_moisture,predicted_soil_moisture
0,0.178049,0.142961
1,0.159784,0.205002
2,0.189298,0.223717
3,0.149692,0.141455
4,0.141275,0.119615
5,0.055885,0.072746
6,0.156202,0.149801
7,0.256181,0.235989
8,0.119157,0.180608
9,0.125611,0.141691


In [13]:
results_df["irrigation_need"] = add_irrigation_category(results_df["predicted_soil_moisture"])
results_df.head(10)

,actual_soil_moisture,predicted_soil_moisture,irrigation_need
0,0.178049,0.142961,HIGH
1,0.159784,0.205002,MEDIUM
2,0.189298,0.223717,LOW
3,0.149692,0.141455,HIGH
4,0.141275,0.119615,HIGH
5,0.055885,0.072746,HIGH
6,0.156202,0.149801,HIGH
7,0.256181,0.235989,LOW
8,0.119157,0.180608,MEDIUM
9,0.125611,0.141691,HIGH


In [14]:
joblib.dump(soil_moisture_model, MODEL_DIR / "soil_moisture_model.joblib")

with open(MODEL_DIR / "soil_moisture_metrics.json", "w") as f:
    json.dump(
        {
            "train": train_metrics,
            "test": test_metrics,
        },
        f,
        indent=2,
    )

results_df.head(20).to_csv(MODEL_DIR / "soil_moisture_predictions.csv", index=False)

print("Saved model and outputs.")

Saved model and outputs.
